In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## مرحلہ 1: ساختہ نتائج کے لیے پائیڈینٹک ماڈلز کی تعریف کریں

یہ ماڈلز وہ **اسکیمہ** متعین کرتے ہیں جو ایجنٹس واپس کریں گے۔ پائیڈینٹک کے ساتھ `response_format` استعمال کرنے کا فائدہ یہ ہے:
- ✅ قسم محفوظ ڈیٹا اخذ کرنا
- ✅ خودکار توثیق
- ✅ آزاد متن کے جوابات سے تجزیہ کی غلطیاں نہیں
- ✅ فیلڈز کی بنیاد پر آسان مشروط راستہ منتخب کرنا


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## مرحلہ 2: ہوٹل بکنگ ٹول بنائیں

یہ ٹول وہ ہے جسے **availability_agent** کال کرے گا تاکہ چیک کیا جا سکے کہ کمرے دستیاب ہیں یا نہیں۔ ہم `@ai_function` ڈیکوریٹر استعمال کرتے ہیں تاکہ:
- ایک Python فنکشن کو AI کال ایبل ٹول میں تبدیل کیا جا سکے
- LLM کے لیے JSON سکیمہ خود بخود جنریٹ ہو
- پیرامیٹرز کی جانچ پڑتال کی جا سکے
- ایجنٹس کی طرف سے خودکار طریقے سے کال کی اجازت دی جا سکے

اس ڈیمو کے لیے:
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → کمروں کی دستیابی ہے ✅
- **تمام دیگر شہر** → کمروں کی دستیابی نہیں ہے ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## مرحلہ 3: روٹنگ کے لیے شرطی فنکشنز کی تعریف کریں

یہ فنکشنز ایجنٹ کے جواب کا معائنہ کرتے ہیں اور طے کرتے ہیں کہ ورک فلو میں کون سا راستہ اختیار کرنا ہے۔

**اہم پیٹرن:**
1. چیک کریں کہ پیغام `AgentExecutorResponse` ہے
2. ساخت شدہ آؤٹ پُٹ (Pydantic ماڈل) کو پارس کریں
3. روٹنگ کو کنٹرول کرنے کے لیے `True` یا `False` واپس کریں

ورک فلو ان شرائط کو **کناروں** پر جانچے گا تاکہ فیصلہ کرے کہ آگے کون سا ایگزیکیٹر چلانا ہے۔


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## مرحلہ 4: کسٹم ڈسپلے ایگزیکیوٹر بنائیں

ایگزیکیوٹرز ورک فلو کے ایسے جزو ہیں جو تبدیلیاں کرتے ہیں یا ضمنی اثرات مرتب کرتے ہیں۔ ہم `@executor` ڈیکوریٹر کا استعمال کرتے ہوئے ایک کسٹم ایگزیکیوٹر بناتے ہیں جو حتمی نتیجہ ظاہر کرتا ہے۔

**اہم تصورات:**
- `@executor(id="...")` - ایک فنکشن کو ورک فلو ایگزیکیوٹر کے طور پر رجسٹر کرتا ہے
- `WorkflowContext[Never, str]` - ان پٹ/آؤٹ پٹ کے لیے ٹائپ ہنٹس
- `ctx.yield_output(...)` - حتمی ورک فلو نتیجہ کو ییلڈ کرتا ہے


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Step 5: لوڈ کریں ماحول کی متغیرات

LLM کلائنٹ کو ترتیب دیں۔ یہ مثال درج ذیل کے ساتھ کام کرتی ہے:
- **GitHub ماڈلز** (GitHub ٹوکن کے ساتھ مفت درجے)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Step 6: منظم نتائج کے ساتھ AI ایجنٹس بنائیں

ہم **تین خصوصی ایجنٹس** بناتے ہیں، ہر ایک کو `AgentExecutor` میں لپیٹا جاتا ہے:

1. **availability_agent** - ٹول کے ذریعے ہوٹل کی دستیابی چیک کرتا ہے  
2. **alternative_agent** - متبادل شہروں کی تجویز دیتا ہے (جب کمرے دستیاب نہ ہوں)  
3. **booking_agent** - بکنگ کی ترغیب دیتا ہے (جب کمرے دستیاب ہوں)  

**اہم خصوصیات:**  
- `tools=[hotel_booking]` - ایجنٹ کو ٹول فراہم کرتا ہے  
- `response_format=PydanticModel` - منظم JSON آؤٹ پٹ کو لازمی بناتا ہے  
- `AgentExecutor(..., id="...")` - ورک فلو استعمال کے لئے ایجنٹ کو لپیٹتا ہے


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## مرحلہ 7: کنڈیشنل ایجز کے ساتھ ورک فلو بنائیں

اب ہم `WorkflowBuilder` استعمال کرتے ہیں تاکہ گراف کو کنڈیشنل راؤٹنگ کے ساتھ تعمیر کیا جا سکے:

**ورک فلو کی ساخت:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**اہم طریقے:**
- `.set_start_executor(...)` - انٹری پوائنٹ سیٹ کرتا ہے
- `.add_edge(from, to, condition=...)` - کنڈیشنل ایج شامل کرتا ہے
- `.build()` - ورک فلو کو حتمی شکل دیتا ہے


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## مرحلہ 8: ٹیسٹ کیس 1 چلائیں - شہر بغیر دستیابی (پیرس)

آئیے **کوئی دستیابی نہیں** راستے کا تجربہ کریں پیرس میں ہوٹلوں کی درخواست دے کر (جس میں ہمارے سمیولیشن میں کوئی کمرے نہیں ہیں).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Step 9: ٹیسٹ کیس 2 چلائیں - شہر دستیابی کے ساتھ (سٹاک ہوم)

اب ہم **دستیابی** راستے کو ٹیسٹ کرتے ہیں اسٹاک ہوم میں ہوٹلوں کی درخواست کر کے (جس میں ہمارے سیمولیشن میں کمرے موجود ہیں).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## اہم نکات اور اگلے اقدامات

### ✅ آپ نے کیا سیکھا:

1. **WorkflowBuilder پیٹرن**
   - داخلے کے نقطہ کو متعین کرنے کے لیے `.set_start_executor()` استعمال کریں
   - شرطی روٹنگ کے لیے `.add_edge(from, to, condition=...)` استعمال کریں
   - ورک فلو کو حتمی شکل دینے کے لیے `.build()` کال کریں

2. **شرطی روٹنگ**
   - کنڈیشن فنکشنز `AgentExecutorResponse` کا معائنہ کرتے ہیں
   - روٹنگ کے فیصلے کرنے کے لیے ساختہ آؤٹ پٹس کو پارس کریں
   - ایک ایج کو فعال کرنے کے لیے `True` واپس کریں، اس سے گزرنے کے لیے `False`

3. **ٹول انٹیگریشن**
   - پائتھن فنکشنز کو AI ٹولز میں تبدیل کرنے کے لیے `@ai_function` استعمال کریں
   - ایجنٹس ضرورت پڑنے پر خودکار طور پر ٹولز کو کال کرتے ہیں
   - ٹولز JSON واپس کرتے ہیں جسے ایجنٹس پارس کر سکتے ہیں

4. **ساختہ آؤٹ پٹس**
   - قسم کی حفاظت کے لیے Pydantic ماڈلز استعمال کریں
   - ایجنٹس بناتے وقت `response_format=MyModel` سیٹ کریں
   - جوابات کو `Model.model_validate_json()` کے ذریعے پارس کریں

5. **حسبِ ضرورت ایکزیگیوٹرز**
   - ورک فلو کے اجزاء بنانے کے لیے `@executor(id="...")` استعمال کریں
   - ایکزیگیوٹرز ڈیٹا کو تبدیل کر سکتے ہیں یا ضمنی اثرات انجام دے سکتے ہیں
   - ورک فلو کے نتائج پیدا کرنے کے لیے `ctx.yield_output()` استعمال کریں

### 🚀 حقیقی دنیا کی درخواستیں:

- **سفر کی بکنگ**: دستیابی چیک کریں، متبادل تجویز کریں، اختیارات کا موازنہ کریں
- **کسٹمر سروس**: مسئلے کی نوعیت، جذبات، ترجیح کی بنیاد پر روٹنگ کریں
- **ای کامرس**: انوینٹری چیک کریں، متبادل تجویز کریں، آرڈرز پراسیس کریں
- **مواد کی نگرانی**: زہریلے پن کے اسکورز، صارف کے جھنڈے کی بنیاد پر روٹنگ کریں
- **تصدیقی ورک فلو**: رقم، صارف کی حیثیت، خطرے کی سطح کے لحاظ سے روٹنگ کریں
- **کئی مراحل کی پروسیسنگ**: ڈیٹا کی معیار، مکمل ہونے کی بنیاد پر روٹنگ کریں

### 📚 اگلے اقدامات:

- مزید پیچیدہ شرائط شامل کریں (متعدد معیار)
- ورک فلو کی حالت کے انتظام کے ساتھ لوپس نافذ کریں
- دوبارہ استعمال کے لیے سب-ورک فلو شامل کریں
- حقیقی APIs سے انٹیگریٹ کریں (ہوٹل بکنگ، انوینٹری سسٹمز)
- غلطی کے ہینڈلنگ اور بیک اپ راستے شامل کریں
- بلٹ ان ویژولائزیشن ٹولز کے ساتھ ورکس فلو کو بصری شکل دیں


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
